# Домашнее задание 3
## Околокруговое приближение

**Введение в маневрирование КА · К лекции 4**  
**Срок сдачи: 19 октября 2026 года, 23:59.**

Исследуйте пассивное движение КА в окрестности круговой орбиты.
Используйте уравнения первого приближения, полученные на лекции.

**Имя и фамилия:** *впишите*  
**Группа:** *впишите*


### Как работать с ноутбуком

Выполняйте ячейки по порядку. Для расчётов нужны Python 3, NumPy и Matplotlib.
В текстовых ячейках оставлены места для аналитических выводов и ответов;
формулы можно записывать в LaTeX между `$…$` или `$$…$$`.
В ячейках параметров заданий 2 и 3 заполните первые четыре строки: начальное отклонение скорости, $a,e_x,e_y$. Остальной код готов к выполнению.

Функции расчёта пассивного движения, построения траекторий и вывода таблиц
уже подготовлены. Они используют решение из методички. Вывод формул
для заданных начальных условий выполните самостоятельно.

В расчётах используйте километры, секунды и радианы.
В таблицах и на рисунках отклонения координат приводятся в метрах,
отклонения скоростей — в м/с.


## Задание 1. Чтение методички

В [методичке `near-circular-transfer.pdf`](../../manual/near-circular-transfer.pdf)
прочитайте разделы **1.1–1.4 и 1.8** главы 1 «Уравнения движения»:
выбор координат, линеаризация, решение на пассивном участке,
параметры орбиты и границы применимости модели.

Дайте краткие обоснованные ответы на следующие вопросы.


### 1.1. Трансверсальная скорость и производная углового отклонения

Чем отличаются отклонение трансверсальной скорости $\Delta V_t$
и величина $r_0\Delta\dot u$? Запишите связь между ними в первом приближении.

> **Подсказка.** Начните с кинематического равенства $V_t=r\dot u$ в разделе 1.1. Подставьте $r=r_0+\Delta r$, $\dot u=n_0+\Delta\dot u$ и $V_t=V_0+\Delta V_t$; раскройте скобки и отбросьте произведение малых отклонений. Используйте $V_0=n_0r_0$ и поясните физический смысл полученных слагаемых.


**Ответ и обоснование:**

*Впишите ответ и формулу.*


### 1.2. Границы применимости

Какие величины должны оставаться малыми для применимости околокругового
приближения? Почему малость начальных отклонений сама по себе не гарантирует
применимости модели на всём интервале движения?

> **Подсказка.** Выпишите безразмерные величины из (1.6), раздел 1.2. Затем рассмотрите (1.12), раздел 1.3: найдите член, растущий со временем при $a\ne r_0$, и сопоставьте его с условием малости $\Delta u$. Объяснение границ применимости приведено в разделе 1.8.


**Ответ и обоснование:**

*Впишите ответ.*


## Общие исходные данные

Опорный аппарат движется по круговой орбите радиуса $r_0$.
Второй аппарат движется в той же плоскости и в том же направлении.
Управление и возмущения отсутствуют: $S=T=W=0$, $z=V_z=0$.

$$
\mu=398600{,}44158\ \text{км}^3/\text{с}^2,\qquad r_0=7000\ \text{км}.
$$
$$
V_0=\sqrt{\frac{\mu}{r_0}},\qquad n_0=\frac{V_0}{r_0},\qquad
T_0=\frac{2\pi}{n_0},\qquad \varphi(t)=n_0t.
$$

В начальный момент аппараты находятся на одном радиальном луче:
$$
\Delta r(0)=100\ \text{м},\qquad \Delta V_r(0)=0,\qquad \Delta u(0)=0.
$$

Здесь $\Delta r=r-r_0$, $\Delta V_t=V_t-V_0$, $\Delta u=u-\varphi$.
Координата $\eta=r_0\Delta u$ описывает продольное смещение в первом приближении.
Величины $\Delta V_r,\Delta V_t$ — отклонения компонент скорости от кругового движения.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

ACCENT = "#187AA5"
SECOND = "#A36540"
INK = "#302B28"
fonts = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams.update({
    "font.family": "PT Serif" if "PT Serif" in fonts else "DejaVu Serif",
    "font.size": 11,
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.titleweight": "normal",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
})


In [ ]:
MU = 398600.44158       # км³/с²
R0 = 7000.0            # км
V0 = np.sqrt(MU / R0)  # км/с
N0 = V0 / R0           # 1/с
T0 = 2 * np.pi / N0    # с

DELTA_R0 = 0.100       # км
DELTA_VR0 = 0.0        # км/с
DELTA_U0 = 0.0         # рад
TIME = np.linspace(0.0, 2 * T0, 1201)
EPOCHS = np.array([0.0, T0, 2 * T0])

print(f"V₀ = {V0:.8f} км/с")
print(f"n₀ = {N0:.10f} 1/с")
print(f"T₀ = {T0:.3f} с = {T0 / 60:.3f} мин")


### Готовые функции

`passive_motion(t, a, ex, ey)` вычисляет плоское пассивное движение
по формулам раздела 1.3 методички при $\varphi(0)=0$.
Аргументы: время в секундах, большая полуось в километрах,
безразмерные $e_x,e_y$. Результат содержит $\Delta r$, $\eta$ в километрах,
$\Delta V_r$, $\Delta V_t$ в км/с и $\Delta u$ в радианах.

`print_state_table` выводит состояние при $t=0,T_0,2T_0$.
`plot_motion` строит траекторию с одинаковым масштабом по пространственным осям.
`print_periodicity_check` выводит разность состояний при $t=T_0$ и $t=0$.
Выполните следующие ячейки; изменять функции для решения задания не требуется.


In [ ]:
def passive_motion(t, a, ex, ey, delta_u0=DELTA_U0):
    """Пассивное движение в первом приближении; φ(0) = 0."""
    t = np.asarray(t, dtype=float)
    phi = N0 * t
    relative_a = (a - R0) / R0
    cos_phi, sin_phi = np.cos(phi), np.sin(phi)
    delta_r = R0 * (relative_a - ex * cos_phi - ey * sin_phi)
    delta_vr = V0 * (ex * sin_phi - ey * cos_phi)
    delta_vt = V0 * (ex * cos_phi + ey * sin_phi - relative_a / 2)
    delta_u = (delta_u0 - 1.5 * relative_a * phi
               + 2 * ex * sin_phi - 2 * ey * (cos_phi - 1))
    return {
        "t": t, "delta_r": delta_r, "delta_vr": delta_vr,
        "delta_vt": delta_vt, "delta_u": delta_u, "eta": R0 * delta_u,
    }


def parameters_ready(*values):
    return all(value is not None for value in values)


def print_parameters(delta_vt0, a, ex, ey, title):
    print(title)
    print(f"ΔVₜ(0) = {delta_vt0 * 1000:.8f} м/с")
    print(f"a = {a:.8f} км;  a − r₀ = {(a - R0) * 1000:.6f} м")
    print(f"eₓ = {ex:.10e};  eᵧ = {ey:.10e}")


def check_initial_state(delta_vt0, a, ex, ey):
    initial = passive_motion(0.0, a, ex, ey)
    actual = [initial["delta_r"], initial["delta_vr"], initial["delta_vt"]]
    expected = [DELTA_R0, DELTA_VR0, delta_vt0]
    if not np.allclose(actual, expected, rtol=1e-5, atol=[1e-6, 1e-9, 1e-9]):
        raise ValueError("Параметры a, eₓ, eᵧ не согласованы с начальными "
                         "координатами и скоростями. Проверьте расчёт и единицы.")


In [ ]:
def print_state_table(orbit, title):
    state = passive_motion(EPOCHS, *orbit)
    print(title)
    print(f"{'t':>5} {'Δr, м':>13} {'η, м':>14} {'ΔVᵣ, м/с':>15} {'ΔVₜ, м/с':>15}")
    for index, label in enumerate(("0", "T₀", "2T₀")):
        values = [state[key][index] * 1000
                  for key in ("delta_r", "eta", "delta_vr", "delta_vt")]
        print(f"{label:>5} {values[0]:13.6f} {values[1]:14.6f} "
              f"{values[2]:15.8f} {values[3]:15.8f}")


def print_periodicity_check(orbit):
    state = passive_motion(np.array([0.0, T0]), *orbit)
    print("Разность состояний: значение при T₀ минус значение при 0")
    for key, label, scale, unit in (
        ("delta_r", "Δr", 1000, "м"),
        ("delta_u", "Δu", 1, "рад"),
        ("delta_vr", "ΔVᵣ", 1000, "м/с"),
        ("delta_vt", "ΔVₜ", 1000, "м/с"),
    ):
        difference = (state[key][1] - state[key][0]) * scale
        print(f"{label}: {difference:+.6e} {unit}")


In [ ]:
def plot_motion(ax, orbit, title, color=ACCENT):
    state = passive_motion(TIME, *orbit)
    ax.plot(state["eta"] * 1000, state["delta_r"] * 1000,
            color=color, linewidth=1.8)
    epochs = passive_motion(EPOCHS, *orbit)

    # Объединяем подписи, если несколько отмеченных положений совпадают.
    groups = []
    for index, label in enumerate(("0", "T_0", "2T_0")):
        point = np.array([epochs["eta"][index], epochs["delta_r"][index]]) * 1000
        for group in groups:
            if np.allclose(point, group["point"], rtol=0, atol=1e-5):
                group["labels"].append(label)
                break
        else:
            groups.append({"point": point, "labels": [label]})

    for group in groups:
        x, y = group["point"]
        ax.scatter(x, y, s=28, color=color, zorder=3)
        ax.annotate("$t = " + ", ".join(group["labels"]) + "$", xy=(x, y),
                    xytext=(0, -10), textcoords="offset points",
                    ha="center", va="top", fontsize=10)

    ax.set(title=title, xlabel=r"$\eta = r_0\Delta u$, м",
           ylabel=r"$\Delta r$, м")
    ax.set_aspect("equal", adjustable="box")
    ax.margins(x=0.10, y=0.45)
    ax.grid(alpha=0.25)


## Задание 2. Равенство начальных угловых скоростей

В начальный момент угловые скорости аппаратов равны:
$$\dot u(0)=n_0,\qquad \Delta\dot u(0)=0.$$

### 2.1. Начальная скорость и параметры орбиты

Найдите $\Delta V_t(0)$ и определите $a,e_x,e_y$ второго аппарата
в первом приближении. Сначала получите аналитические выражения,
затем вычислите численные значения.

> **Подсказка.** Используйте связь скоростей из пункта 1 задания 1 и условие $\Delta\dot u(0)=0$. Для $a,e_x,e_y$ возьмите (1.13) и (1.15) из раздела 1.4; при $t=0$ имеем $\varphi=0$, поэтому $\cos\varphi=1$, $\sin\varphi=0$. Нормировки даны в (1.9). Числа подставляйте после упрощения формул, переведя $100$ м в $0{,}1$ км.


**Аналитическое решение:**

*Запишите связь начальных скоростей, затем выведите выражения для
$\Delta V_t(0)$ и $a,e_x,e_y$.*


In [ ]:
# Впишите найденные выражения; используйте константы из ячейки исходных данных.
delta_vt0_2 = None  # км/с
a_2 = None          # км
ex_2 = None         # безразмерная величина
ey_2 = None         # безразмерная величина

orbit_2 = None
if parameters_ready(delta_vt0_2, a_2, ex_2, ey_2):
    check_initial_state(delta_vt0_2, a_2, ex_2, ey_2)
    orbit_2 = (a_2, ex_2, ey_2)
    print_parameters(delta_vt0_2, *orbit_2, "Задание 2")
else:
    print("Заполните ΔVₜ(0), a, eₓ, eᵧ для задания 2.")


### 2.2. Закон движения и траектория

Подставьте найденные параметры в решение пассивного движения из методички.
Получите выражения для $\Delta r(t)$ и $\Delta u(t)$.
Постройте траекторию на интервале $0\le t\le2T_0$ в координатах $(\eta,\Delta r)$.
По горизонтальной оси отложите $\eta$, по вертикальной — $\Delta r$;
положительное направление $\eta$ совпадает с направлением движения
опорного аппарата. Отметьте положения при $t=0,T_0,2T_0$.

> **Подсказка.** Возьмите выражение для $\Delta\widehat r$ из (1.11) и для $\Delta u$ из (1.12), раздел 1.3. Подставьте найденные $a,e_x,e_y$ и положите $\varphi_*=0$, $\Delta u(\varphi_*)=0$, $\varphi=n_0t$. Вернитесь к размерной координате $\Delta r=r_0\Delta\widehat r$, затем найдите $\eta=r_0\Delta u$. Проверьте формулы при $t=0$; график строит готовая ячейка ноутбука.


**Подстановка и полученные выражения:**

*Приведите выражения для $\Delta r(t)$, $\Delta u(t)$ и $\eta(t)$
для начальных условий задания 2.*


In [ ]:
if orbit_2 is not None:
    fig, ax = plt.subplots(figsize=(11, 3.8), constrained_layout=True)
    plot_motion(ax, orbit_2, "Задание 2. Равенство начальных угловых скоростей")
    plt.show()
else:
    print("Сначала заполните и выполните ячейку параметров задания 2.")


### 2.3. Состояние после одного и двух периодов

Составьте таблицу $\Delta r,\eta,\Delta V_r,\Delta V_t$ при $t=0,T_0,2T_0$.
Сохраняется ли равенство угловых скоростей при дальнейшем движении?
Объясните знак и накопление продольного смещения,
связав их с большой полуосью орбиты второго аппарата.

> **Подсказка.** Для таблицы подставьте $\varphi=0,2\pi,4\pi$ в (1.11), (1.12); скорости переведите из нормированного вида по (1.9). Найдите $\Delta\dot u(t)$ по (1.7) или дифференцированием своего ответа. По формуле $\Delta u(t)$ определите знак накопленного смещения; его физический смысл объясните через среднее движение $n=\sqrt{\mu/a^3}$.


In [ ]:
if orbit_2 is not None:
    print_state_table(orbit_2, "Задание 2")
else:
    print("Сначала заполните и выполните ячейку параметров задания 2.")


**Объяснение результатов:**

*Ответьте на вопрос об угловых скоростях. Объясните знак продольного
смещения и его изменение от одного периода к следующему.*


## Задание 3. Периодическое движение

Сохраните заданные $\Delta r(0),\Delta V_r(0),\Delta u(0)$,
а $\Delta V_t(0)$ выберите из условия периодичности движения
относительно опорного аппарата.

### 3.1. Условие периодичности

Найдите значение $\Delta V_t(0)$, при котором в линейной модели
состояние $\Delta r,\Delta u,\Delta V_r,\Delta V_t$ повторяется через период $T_0$.
Обоснуйте условие периодичности по выражению для $\Delta u(t)$
и определите соответствующие $a,e_x,e_y$.

> **Подсказка.** В формуле (1.12) сравните $\Delta u(\varphi+2\pi)$ и $\Delta u(\varphi)$. Тригонометрические члены повторяются; требование нулевого приращения за период даст условие на $a$. Подставьте его в (1.13) и найдите $\Delta V_t(0)$. Затем вычислите $e_x,e_y$ по (1.15) при $\varphi=0$.


**Аналитическое решение:**

*Выведите условие периодичности. Найдите $\Delta V_t(0)$ и $a,e_x,e_y$.*


In [ ]:
delta_vt0_3 = None  # км/с
a_3 = None          # км
ex_3 = None         # безразмерная величина
ey_3 = None         # безразмерная величина

orbit_3 = None
if parameters_ready(delta_vt0_3, a_3, ex_3, ey_3):
    check_initial_state(delta_vt0_3, a_3, ex_3, ey_3)
    orbit_3 = (a_3, ex_3, ey_3)
    print_parameters(delta_vt0_3, *orbit_3, "Задание 3")
else:
    print("Заполните ΔVₜ(0), a, eₓ, eᵧ для задания 3.")


### 3.2. Траектории и проверка периодичности

Получите выражения для $\Delta r(t)$ и $\Delta u(t)$.
Постройте траектории из заданий 2 и 3 на одном рисунке в двух панелях,
с одинаковым масштабом по осям $\eta$ и $\Delta r$ внутри каждой панели.
Отметьте положения при $t=0,T_0,2T_0$.
Дополните таблицу результатами для задания 3 и проверьте повторение
всех четырёх компонент состояния через $T_0$.

> **Подсказка.** Повторите подстановку в (1.11), (1.12) с параметрами задания 3. Проверьте каждую компоненту состояния при $\varphi=0$ и $2\pi$, в том числе $\Delta u$. Используйте готовые ячейки с двумя графиками, таблицами и проверкой периодичности. Малые ненулевые разности сопоставьте с погрешностью округления.


**Подстановка и полученные выражения:**

*Приведите выражения для $\Delta r(t)$, $\Delta u(t)$ и $\eta(t)$
для начальных условий задания 3.*


In [ ]:
if orbit_2 is not None and orbit_3 is not None:
    fig, axes = plt.subplots(2, 1, figsize=(11, 8), constrained_layout=True)
    plot_motion(axes[0], orbit_2, "Задание 2. Равенство начальных угловых скоростей")
    plot_motion(axes[1], orbit_3, "Задание 3. Периодическое движение", SECOND)
    plt.show()
else:
    print("Сначала заполните и выполните ячейки параметров заданий 2 и 3.")


In [ ]:
if orbit_2 is not None and orbit_3 is not None:
    print_state_table(orbit_2, "Задание 2")
    print()
    print_state_table(orbit_3, "Задание 3")
    print()
    print_periodicity_check(orbit_3)
else:
    print("Сначала заполните и выполните ячейки параметров заданий 2 и 3.")


**Проверка периодичности:**

*Сопоставьте численные разности состояний с аналитическим решением.*


### 3.3. Сравнение результатов

Объясните, почему равенство начальных угловых скоростей и периодичность
движения соответствуют разным значениям $\Delta V_t(0)$.
Какую роль в этом различии играет большая полуось?

> **Подсказка.** Сопоставьте найденные $\Delta V_t(0)$ и $a$ в двух заданиях. Объясните, какое условие относится только к начальному моменту, а какое — к движению за целый период. Свяжите различие с наличием или отсутствием линейного по $\varphi$ члена в (1.12).


**Ответ:**

*Сравните полученные начальные скорости, большие полуоси и характер движения.*


## Что представить

Сдайте этот ноутбук с ответами на вопросы задания 1 и полными решениями заданий 2 и 3:
аналитическими выражениями, численными значениями начальных скоростей
и параметров орбит, траекториями, таблицами и объяснением результатов.
Формулы и рассуждения оформите в текстовых ячейках, расчёты и рисунки — в кодовых.

Перед отправкой перезапустите ядро, выполните все ячейки по порядку
и сохраните ноутбук с результатами вычислений и графиками.

Название файла: **`hw_3_name_surname.ipynb`**.
Вместо `name_surname` укажите свои имя и фамилию латиницей.
